In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [10]:
# Environment setup
grid_size = (4, 4)
terminal_states = [(0, 0), (3, 3)]
rewards = -0.1 * np.ones(grid_size)
rewards[0, 0] = 1.0
rewards[3, 3] = 1.0
gamma = 0.9
episodes = 1
actions = ['U', 'D', 'L', 'R']
action_to_delta = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}

In [11]:
def get_next_state(state, action):
    delta = action_to_delta[action]
    next_state = (state[0] + delta[0], state[1] + delta[1])
    if 0 <= next_state[0] < grid_size[0] and 0 <= next_state[1] < grid_size[1]:
        return next_state
    return state

def generate_episode(policy):
    state = (random.randint(0, 3), random.randint(0, 3))
    while state in terminal_states:
        state = (random.randint(0, 3), random.randint(0, 3))
    episode = []
    while True:
        action = policy[state]
        next_state = get_next_state(state, action)
        reward = rewards[next_state]
        episode.append((state, action, reward))
        if next_state in terminal_states:
            break
        state = next_state
    return episode

def mc_policy_iteration_sampling():
    Q = np.zeros(grid_size + (len(actions),))
    returns = [[{a: [] for a in range(len(actions))} for _ in range(grid_size[1])] for _ in range(grid_size[0])]
    policy = {(i, j): random.choice(actions) for i in range(grid_size[0]) for j in range(grid_size[1])
              if (i, j) not in terminal_states}

    for ep in range(episodes):
        episode = generate_episode(policy)
        G = 0
        visited = set()
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward
            a_idx = actions.index(action)
            if (state, action) not in visited:
                returns[state[0]][state[1]][a_idx].append(G)
                Q[state[0], state[1], a_idx] = np.mean(returns[state[0]][state[1]][a_idx])
                visited.add((state, action))

        for (state, action, _) in episode:
            if state in terminal_states:
                continue
            best_action_idx = np.argmax(Q[state[0], state[1]])
            policy[state] = actions[best_action_idx]

    return Q, policy

In [12]:
def plot_policy(Q, policy, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    V = np.max(Q, axis=-1)
    arrow_dict = {'U': (0, 0.3), 'D': (0, -0.3), 'L': (-0.3, 0), 'R': (0.3, 0)}

    for i in range(grid_size[0]):
        for j in range(grid_size[1]):
            state = (i, j)
            ax.add_patch(patches.Rectangle((j, grid_size[0] - 1 - i), 1, 1, fill=False))
            ax.text(j + 0.5, grid_size[0] - 1 - i + 0.4, f"{V[i, j]:.2f}", ha='center', fontsize=8)
            if state in policy:
                direction = policy[state]
                dx, dy = arrow_dict[direction]
                ax.arrow(j + 0.5 - dx / 2, grid_size[0] - 1 - i + 0.5 - dy / 2, dx, dy,
                         head_width=0.1, head_length=0.1, fc='black', ec='black')
            else:
                ax.text(j + 0.5, grid_size[0] - 1 - i + 0.25, 'T', ha='center', fontsize=12)

    ax.set_xlim(0, grid_size[1])
    ax.set_ylim(0, grid_size[0])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run and plot
Q_mc, policy_mc = mc_policy_iteration_sampling()
plot_policy(Q_mc, policy_mc, "Monte Carlo Policy Iteration (Improvement Sampling)")